# 04. Model results: tuning, the core against full ablation, and host leakage

This notebook **fits nothing**. Every number in it is read from the checkpoint
files written by `scripts/train_experiments.py`, so it runs in a minute or two.

That separation is not tidiness, it is what made the work finish. Training is
memory bound, so it lives in a script that runs one search at a time and writes
each result to disk before starting the next. Section 8 tells that story,
including the three runs that failed and the silent `nan` that made a failure
look like a result.

What the training did. The modelling population is the listings with
`long_stay == False`. The split is 80/20 by `GroupShuffleSplit` on `host_id`,
so no account appears on both sides: one account can own dozens of listings
priced to the same formula, and a random split would let a model score well by
recalling an account's habits rather than by pricing the market. Section 6
measures exactly that. The target is `log1p(price_num)`, with every dollar
figure obtained by mapping predictions back with `expm1`. Tukey fences
(k=1.5) on log price are learned from the training rows only; models train
inside them and are scored on two test sets, `typical` inside the fences and
`all` unfiltered. Three models, three feature sets, 10-fold `GridSearchCV`
throughout, every fit with `error_score="raise"` so a failure stops rather than
averaging into a `nan`.

Sections:

1. Scope, split and fences
2. Results table
3. Winners, core against full, and the space ablation
4. Grid edge flags
5. Importances and group shares
6. Host leakage: grouped against random split
7. Figures
8. Run history: three failures and what changed
9. Run 1 against this run
10. Save `reports/model_results.json`
11. Learning curve (extra)
12. Out-of-fold residual quantiles (extra)
13. Permutation importance (extra)
14. Standard against workstation profile
15. Verification

Sections 11 to 13 appear only when `scripts/train_experiments.py --extras` has
been run, and section 14 only when both profiles have checkpoints. Each one
skips quietly otherwise.

## 0. Setup

Only the checkpoints and the dataset are read. `scripts.train_experiments` is
imported for its `Experiment` class, which rebuilds the split and the fences
from the same code the training used, so the notebook and the script cannot
disagree about which rows are in which test set. Rebuilding a split is index
arithmetic; nothing is fitted.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import scripts.train_experiments as te  # noqa: E402
from src.config import REPORTS_DIR  # noqa: E402

# Which checkpoint folder to read. The script keeps one per profile, and
# notebook 05 passes the profile through the environment, so a workstation run
# renders the workstation checkpoints and a plain Run All renders the standard
# ones. Reading this from the script rather than hardcoding a path is what
# keeps the two from disagreeing about which run is being reported.
CONFIG = te.configure_from_env()
CHECKPOINT_DIR = CONFIG.checkpoints
CURRENCY = te.CURRENCY
UNITS = te.UNITS
Experiment = te.Experiment
checkpoint_path = te.checkpoint_path
predictions_path = te.predictions_path

# Smoke runs get their own figure folder and their own results file. Without
# this a smoke run overwrites reports/figures/model_*.png and
# reports/model_results.json with 3-fold numbers from a tenth of the rows,
# which is the kind of quietly damaged output this project has been bitten by
# before. The checkpoints were already separated by folder; these were not.
SUFFIX = "_smoke" if CONFIG.smoke else ""
FIGURE_DIR = REPORTS_DIR / f"figures{SUFFIX}"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = REPORTS_DIR / f"model_results{SUFFIX}.json"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 150)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9.0, 5.0)
plt.rcParams["figure.dpi"] = 80
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"

SET_ORDER = ["core", "core_plus_space", "full"]
MODEL_ORDER = ["Ridge", "RandomForest", "XGBoost"]

context = json.loads((CHECKPOINT_DIR / "_context.json").read_text(encoding="utf-8"))

checkpoints = {}
missing = []
for model_name, set_name in UNITS:
    path = checkpoint_path(model_name, set_name)
    if path.exists():
        checkpoints[(model_name, set_name)] = json.loads(
            path.read_text(encoding="utf-8")
        )
    else:
        missing.append(f"{model_name}/{set_name}")

extras = {}
for extra_name in te.EXTRA_UNITS:
    extra_file = te.extra_path(extra_name)
    if extra_file.exists():
        extras[extra_name] = json.loads(extra_file.read_text(encoding="utf-8"))

leakage_path = checkpoint_path("leakage", "all")
leakage_record = (
    json.loads(leakage_path.read_text(encoding="utf-8"))
    if leakage_path.exists()
    else None
)

print(f"profile           : {CONFIG.profile}"
      f"{'  (SMOKE, not a real result)' if CONFIG.smoke else ''}")
print(f"checkpoint folder : reports/{CHECKPOINT_DIR.name}")
print(f"checkpoints found : {len(checkpoints)} of {len(UNITS)} search units")
print(f"leakage unit      : {'present' if leakage_record else 'MISSING'}")
if missing:
    print(f"MISSING units     : {missing}")
else:
    print("every search unit is present")

print(f"extras present    : {sorted(extras) or 'none'}")

print()
print(f"Random Forest max_features : {context['rf_max_features']['selected']}")
print(f"  rule   : {context['rf_max_features']['rule']}")
print(f"  source : {context['rf_max_features']['source']}")

## 1. Scope, split and fences

All of this is read from the context file the training script wrote, so these
are the figures the models were actually trained against.

In [ ]:
scope = context["scope"]
split = context["split"]
fences = context["fences"]

print("scope")
print(f"  master rows                 : {scope['master_rows']:,}")
print(f"  excluded, long_stay True    : {scope['rows_excluded_long_stay']:,}")
print(f"  excluded, long_stay unknown : "
      f"{scope['rows_excluded_long_stay_unknown']:,}")
print(f"  rows modelled               : {scope['rows_modelled']:,}")

print()
print("split")
print(f"  method        : {split['method']}")
print(f"  train rows    : {split['train_rows']:,} across "
      f"{split['train_hosts']:,} hosts")
print(f"  test rows     : {split['test_rows']:,} across "
      f"{split['test_hosts']:,} hosts")
print(f"  hosts in both : {split['hosts_in_both']}")

print()
print(f"fences on {fences['scale']}, k={fences['k']}, {fences['fitted_on']}")
print(f"  Q1    = {fences['q1_log']:.4f}   Q3 = {fences['q3_log']:.4f}")
print(f"  lower = {fences['lower_log']:.4f}  ->  "
      f"{fences['lower_aud']:,.2f} {CURRENCY}")
print(f"  upper = {fences['upper_log']:.4f}  ->  "
      f"{fences['upper_aud']:,.2f} {CURRENCY}")
print()
print("rows outside the fences")
print(pd.DataFrame(fences["rows_outside"]).T.to_string())
print()
print(f"  training rows used : {fences['training_rows_used']:,}")
print(f"  test set 'typical' : {fences['test_rows_typical']:,}")
print(f"  test set 'all'     : {fences['test_rows_all']:,}")

print()
print("cross-validation")
for key, value in context["cv"].items():
    print(f"  {key:<8}: {value}")

print()
print("feature sets")
for name in SET_ORDER:
    print(f"  {name:<16} {len(context['feature_sets'][name]):>2} columns")

## 2. Results table

CV log RMSE is the selection criterion, measured on the scale the models are
fitted on and without touching the test set. The dollar metrics are held out,
reported on both test sets.

`typical` is the honest headline for a pricing tool, because it covers the
price range the model was trained to serve. `all` includes the listings above
the upper fence that no model saw in training, so it is where the cost of
filtering shows up rather than being hidden.

In [ ]:
rows = []
for (model_name, set_name), record in checkpoints.items():
    row = {
        "model": model_name,
        "feature_set": set_name,
        "n_columns": record["n_columns"],
        "n_encoded": record["n_encoded"],
        "best_params": record["best_params"],
        "cv_rmse_log_mean": record["cv_rmse_log_mean"],
        "cv_rmse_log_sd": record["cv_rmse_log_sd"],
        "fit_seconds": record["wall_seconds"],
        "artefact_mb": round(record["artefact_bytes"] / 1e6, 1),
    }
    for test_name, metrics in record["metrics"].items():
        for metric, value in metrics.items():
            row[f"{metric}_{test_name}"] = value
    rows.append(row)

results = pd.DataFrame(rows)
results["_set"] = pd.Categorical(results["feature_set"], SET_ORDER, ordered=True)
results["_model"] = pd.Categorical(results["model"], MODEL_ORDER, ordered=True)
results = (
    results.sort_values(["_set", "_model"])
    .drop(columns=["_set", "_model"])
    .reset_index(drop=True)
)

print(f"all figures in {CURRENCY} unless marked log. "
      f"typical n={fences['test_rows_typical']:,}, "
      f"all n={fences['test_rows_all']:,}")
print()
print(
    results[["model", "feature_set", "n_columns", "n_encoded",
             "cv_rmse_log_mean", "cv_rmse_log_sd",
             "rmse_aud_typical", "mae_aud_typical", "r2_typical",
             "rmse_log_typical",
             "rmse_aud_all", "mae_aud_all", "r2_all", "rmse_log_all",
             "fit_seconds", "artefact_mb"]]
    .round(4)
    .to_string(index=False)
)

print()
print("best parameters")
for row in results.itertuples(index=False):
    print(f"  {row.model:<14} {row.feature_set:<16} {row.best_params}")

In [ ]:
print(f"=== scored on the TYPICAL test set "
      f"(n={fences['test_rows_typical']:,}) ===")
print(
    results[["model", "feature_set", "cv_rmse_log_mean", "rmse_aud_typical",
             "mae_aud_typical", "r2_typical", "rmse_log_typical"]]
    .rename(columns={"cv_rmse_log_mean": "CV RMSE (log)",
                     "rmse_aud_typical": f"RMSE ({CURRENCY})",
                     "mae_aud_typical": f"MAE ({CURRENCY})",
                     "r2_typical": "R2",
                     "rmse_log_typical": "RMSE (log)"})
    .round(4)
    .to_string(index=False)
)

print()
print(f"=== scored on the ALL test set (n={fences['test_rows_all']:,}) ===")
print(
    results[["model", "feature_set", "rmse_aud_all", "mae_aud_all", "r2_all",
             "rmse_log_all"]]
    .rename(columns={"rmse_aud_all": f"RMSE ({CURRENCY})",
                     "mae_aud_all": f"MAE ({CURRENCY})",
                     "r2_all": "R2",
                     "rmse_log_all": "RMSE (log)"})
    .round(4)
    .to_string(index=False)
)

print()
print("CV log RMSE by model and feature set")
pivot = results.pivot_table(index="model", columns="feature_set",
                            values="cv_rmse_log_mean")
pivot = pivot[[c for c in SET_ORDER if c in pivot.columns]]
if {"core", "full"} <= set(pivot.columns):
    pivot["full_vs_core_pct"] = 100 * (pivot["full"] - pivot["core"]) / pivot["core"]
if {"core", "core_plus_space"} <= set(pivot.columns):
    pivot["space_vs_core_pct"] = (
        100 * (pivot["core_plus_space"] - pivot["core"]) / pivot["core"]
    )
print(pivot.round(4).to_string())

## 3. Winners, core against full, and the space ablation

The winner for each feature set is chosen on CV log RMSE.

**core against full is the hypothesis.** `core` is what the application can
supply for a mentor with no history. `full` adds the platform-derived
performance, rating, availability and demand columns, which a new mentor does
not have. The gap is what those columns are worth.

**core_plus_space is a benchmark, not a candidate.** `bedrooms`, `beds` and
`bathrooms` are physical-property attributes with no mentorship equivalent, so
they can never reach the application. Their gain over `core` measures how much
of the achievable accuracy on this proxy dataset comes from property size,
which is accuracy the real system cannot have.

In [ ]:
winners = {}
for set_name in SET_ORDER:
    subset = results.loc[results["feature_set"] == set_name]
    if len(subset):
        winners[set_name] = subset.loc[subset["cv_rmse_log_mean"].idxmin()]

print("winner by feature set, selected on CV log RMSE")
print()
for set_name, row in winners.items():
    print(f"{set_name}")
    print(f"  model       : {row['model']}")
    print(f"  best params : {row['best_params']}")
    print(f"  CV log RMSE : {row['cv_rmse_log_mean']:.4f} "
          f"+/- {row['cv_rmse_log_sd']:.4f}")
    print(f"  typical     : RMSE {row['rmse_aud_typical']:>8,.2f} {CURRENCY}  "
          f"MAE {row['mae_aud_typical']:>7,.2f}  "
          f"R2 {row['r2_typical']:.4f}  log {row['rmse_log_typical']:.4f}")
    print(f"  all         : RMSE {row['rmse_aud_all']:>8,.2f} {CURRENCY}  "
          f"MAE {row['mae_aud_all']:>7,.2f}  "
          f"R2 {row['r2_all']:.4f}  log {row['rmse_log_all']:.4f}")
    print()


def compare(baseline_name, variant_name):
    """Variant against baseline, on the winner of each."""
    base, variant = winners[baseline_name], winners[variant_name]
    out = []
    for metric, label, lower_is_better in (
        ("cv_rmse_log_mean", "CV log RMSE", True),
        ("rmse_log_typical", "log RMSE (typical)", True),
        ("rmse_aud_typical", f"RMSE {CURRENCY} (typical)", True),
        ("mae_aud_typical", f"MAE {CURRENCY} (typical)", True),
        ("r2_typical", "R2 (typical)", False),
        ("rmse_log_all", "log RMSE (all)", True),
        ("rmse_aud_all", f"RMSE {CURRENCY} (all)", True),
        ("r2_all", "R2 (all)", False),
    ):
        change = variant[metric] - base[metric]
        better = (change < 0) if lower_is_better else (change > 0)
        out.append(
            {
                "metric": label,
                baseline_name: base[metric],
                variant_name: variant[metric],
                "change": change,
                "change_pct": 100 * change / base[metric] if base[metric] else np.nan,
                "variant_better": better,
            }
        )
    return pd.DataFrame(out)


core_vs_full = pd.DataFrame()
core_vs_space = pd.DataFrame()

if {"core", "full"} <= set(winners):
    print("=" * 78)
    print("core against full: the hypothesis")
    print("=" * 78)
    core_vs_full = compare("core", "full")
    print(core_vs_full.round(4).to_string(index=False))
    cv_gap = (winners["full"]["cv_rmse_log_mean"]
              - winners["core"]["cv_rmse_log_mean"])
    print()
    print(f"winner on core : {winners['core']['model']}")
    print(f"winner on full : {winners['full']['model']}")
    print(f"CV log RMSE gap: {cv_gap:+.4f} "
          f"({100 * cv_gap / winners['core']['cv_rmse_log_mean']:+.2f}%)")
    print(f"typical MAE gap: "
          f"{winners['full']['mae_aud_typical'] - winners['core']['mae_aud_typical']:+,.2f} "
          f"{CURRENCY}")

if {"core", "core_plus_space"} <= set(winners):
    print()
    print("=" * 78)
    print("core_plus_space against core: how much comes from property size")
    print("=" * 78)
    core_vs_space = compare("core", "core_plus_space")
    print(core_vs_space.round(4).to_string(index=False))
    space_gap = (winners["core_plus_space"]["cv_rmse_log_mean"]
                 - winners["core"]["cv_rmse_log_mean"])
    print()
    print(f"CV log RMSE gap: {space_gap:+.4f} "
          f"({100 * space_gap / winners['core']['cv_rmse_log_mean']:+.2f}%)")
    print()
    print("these three columns are not available to the application, so any")
    print("gain here is accuracy the deployed system cannot have")

## 4. Grid edge flags

A best value sitting at the end of its range means the search wanted to keep
going that way, so the reported optimum may not be one. Interior values mean
the grid bracketed the optimum.

The XGBoost grid was widened for this run precisely because the first run put
`n_estimators` and `max_depth` on their upper edges in all three feature sets.
Section 8 has the detail.

In [ ]:
edge_rows = []
for (model_name, set_name), record in checkpoints.items():
    for edge in record["grid_edges"]:
        edge_rows.append({"model": model_name, "feature_set": set_name, **edge})
edges = pd.DataFrame(edge_rows)
edges["_set"] = pd.Categorical(edges["feature_set"], SET_ORDER, ordered=True)
edges["_model"] = pd.Categorical(edges["model"], MODEL_ORDER, ordered=True)
edges = (
    edges.sort_values(["_set", "_model", "parameter"])
    .drop(columns=["_set", "_model"])
    .reset_index(drop=True)
)
edges["candidates"] = edges["candidates"].apply(str)
print(edges.to_string(index=False))

flagged = edges.loc[edges["position"].str.contains("EDGE")]
print()
print(f"{len(flagged)} of {len(edges)} tuned parameters sit on a grid edge")
if len(flagged):
    print()
    print("flagged")
    for row in flagged.itertuples(index=False):
        print(f"  {row.model:<14} {row.feature_set:<16} {row.parameter:<18} "
              f"chose {row.chosen:<6} from {row.candidates:<26} -> {row.position}")
    print()
    print("where a value sits on an edge the grid did not bracket the optimum,")
    print("so that model's tuned score is a lower bound on what it could reach")
else:
    print("no parameter sits on an edge: every grid bracketed its optimum")

print()
print("by model")
print(
    edges.assign(on_edge=edges["position"].str.contains("EDGE"))
    .groupby("model", observed=True)["on_edge"]
    .agg(parameters="size", on_edge="sum")
    .to_string()
)

## 5. Importances and group shares

Gain for XGBoost, impurity decrease for Random Forest, absolute standardised
coefficient for Ridge. These are not comparable across model families, so each
is read within its own model.

The group shares answer what a single list cannot: of the fit the `full` model
achieves, how much is location, how much is track record, and how much is
demand. One name is a trap, so the assignment is printed in full:
`review_scores_location` is a guest rating, which makes it performance, not
location.

In [ ]:
GROUPS = context["importance_groups"]
print(f"group assignment, {sum(len(v) for v in GROUPS.values())} columns")
print()
for group_name, columns in GROUPS.items():
    print(f"{group_name} ({len(columns)})")
    for column in columns:
        print(f"  {column}")
    print()
print("note: review_scores_location is a guest rating, so it is performance,")
print("not location. Location is carried by neighbourhood_cleansed alone.")


def importance_frame(model_name, set_name):
    record = checkpoints[(model_name, set_name)]["importances"]
    frame = pd.DataFrame(record["rows"])
    frame.attrs["kind"] = record["kind"]
    return frame


importances = {}
for set_name in ("core", "full"):
    if set_name not in winners:
        continue
    model_name = winners[set_name]["model"]
    table = importance_frame(model_name, set_name)
    importances[set_name] = table
    print()
    print("=" * 78)
    print(f"top 10 importances, {set_name} winner: {model_name} "
          f"({table.attrs['kind']})")
    print("=" * 78)
    print(
        table.head(10)[["rank", "feature", "source_column", "group",
                        "importance", "share_pct"]]
        .round(5)
        .to_string(index=False)
    )

In [ ]:
group_shares = pd.DataFrame()
headline = {}
by_source = pd.DataFrame()

if "full" in importances:
    full_table = importances["full"]
    group_shares = (
        full_table.groupby("group")["share_pct"]
        .agg(share_pct="sum", n_encoded="size")
        .sort_values("share_pct", ascending=False)
        .round(2)
    )
    group_shares["n_source_columns"] = (
        full_table.groupby("group")["source_column"].nunique()
    )
    print(f"group shares of total importance, full model "
          f"({winners['full']['model']}, {full_table.attrs['kind']})")
    print()
    print(group_shares.to_string())

    headline = {
        group: float(group_shares.loc[group, "share_pct"])
        if group in group_shares.index
        else 0.0
        for group in ("location", "performance", "demand")
    }
    print()
    print("the three groups asked for")
    for group, share in headline.items():
        print(f"  {group:<12} {share:6.2f}%")
    other = (
        float(group_shares.loc["profile_other", "share_pct"])
        if "profile_other" in group_shares.index
        else 0.0
    )
    print(f"  {'(profile_other)':<12} {other:6.2f}%")
    print()
    print(f"  the three together: {sum(headline.values()):.2f}%")

    by_source = (
        full_table.groupby(["source_column", "group"])["share_pct"]
        .sum()
        .reset_index()
        .sort_values("share_pct", ascending=False, ignore_index=True)
    )
    print()
    print("top source columns in the full model, summed over encodings")
    print(by_source.head(15).round(3).to_string(index=False))

if "core" in importances:
    print()
    print(f"group shares, core model ({winners['core']['model']})")
    print(
        importances["core"].groupby("group")["share_pct"].sum()
        .sort_values(ascending=False).round(2).to_string()
    )

## 6. Host leakage: grouped against random split

XGBoost at its tuned parameters, refitted on the grouped split and on a plain
random split of the same data. Each arm takes its fences from its own training
rows, so neither borrows a filtering decision made with the other's data.

The two numbers are not measured on the same test set, and that is the point.
The random split's test rows include listings whose owner is also in training,
so if its score is better, the difference is the model recalling a host's
pricing rather than pricing the market. Any gap is an upper bound on how much a
random split would have flattered this project.

In [ ]:
leakage = pd.DataFrame()
leakage_gap = pd.DataFrame()

if leakage_record is None:
    print("the leakage unit has no checkpoint yet")
else:
    leakage = pd.DataFrame(leakage_record["rows"])
    leakage_gap = pd.DataFrame(leakage_record["gaps"])
    print(leakage_record["method"])
    print()
    print(leakage.round(4).to_string(index=False))

    print()
    print("random split minus grouped split")
    print(leakage_gap.round(4).to_string(index=False))

    print()
    print("a negative rmse_log_change means the random split looked BETTER,")
    print("which is the signature of host leakage inflating the score")
    worst = leakage_gap.loc[leakage_gap["rmse_log_change_pct"].idxmin()]
    print()
    print(f"largest inflation: {worst['feature_set']} at "
          f"{worst['rmse_log_change_pct']:+.2f}% log RMSE "
          f"({worst['rmse_log_grouped']:.4f} grouped against "
          f"{worst['rmse_log_random']:.4f} random)")
    random_rows = leakage.loc[leakage["split"] == "random"].iloc[0]
    print(f"on the random split, {random_rows['test_rows_with_seen_host']:,} "
          f"test rows belong to a host also seen in training, and "
          f"{random_rows['hosts_in_both']:,} hosts straddle the boundary")

## 7. Figures

Three figures, each covering both winners so the core against full comparison
is visible rather than described. The predictions come from the `.npz` files
the training script saved, so nothing is refitted here.

In [ ]:
# Rebuilding the split is index arithmetic on the parquet, not model fitting.
# Reusing the script's own class guarantees the indices match the saved
# predictions rather than merely resembling them.
experiment = Experiment()
actuals = {
    name: experiment.rows["price_num"].iloc[index].to_numpy()
    for name, index in experiment.test_sets.items()
}
print("actuals rebuilt from the dataset:")
for name, values in actuals.items():
    print(f"  {name:<8} {len(values):,} rows, "
          f"{values.min():,.2f} to {values.max():,.2f} {CURRENCY}")


def stored_predictions(set_name, test_name):
    model_name = winners[set_name]["model"]
    with np.load(predictions_path(model_name, set_name)) as handle:
        return handle[test_name]


for set_name in importances:
    for test_name, actual in actuals.items():
        predicted = stored_predictions(set_name, test_name)
        assert len(predicted) == len(actual), (
            f"{set_name}/{test_name}: {len(predicted)} predictions against "
            f"{len(actual)} actuals"
        )
print()
print("stored predictions line up with the rebuilt test sets")


def save_figure(fig, name):
    fig.savefig(FIGURE_DIR / f"{name}.png")
    print(f"saved reports/figures/{name}.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))
for axis, set_name in zip(axes, ("core", "full")):
    actual = actuals["typical"]
    predicted = stored_predictions(set_name, "typical")
    axis.scatter(actual, predicted, s=5, alpha=0.2, color="#4C72B0", rasterized=True)
    limit = [actual.min(), actual.max()]
    axis.plot(limit, limit, color="#C44E52", lw=1.5, label="perfect prediction")
    axis.set_xscale("log")
    axis.set_yscale("log")
    axis.set_xlabel(f"actual price ({CURRENCY}, log scale)")
    axis.set_ylabel(f"predicted price ({CURRENCY}, log scale)")
    axis.set_title(f"{set_name}: {winners[set_name]['model']}\n"
                   f"typical test set, MAE "
                   f"{winners[set_name]['mae_aud_typical']:,.0f} {CURRENCY}")
    axis.legend(loc="upper left", fontsize=8)
fig.suptitle("Predicted against actual price, 80/20 split grouped by host")
fig.tight_layout()
save_figure(fig, "model_predicted_vs_actual")
plt.show()

In [ ]:
PRICE_BANDS = [0, 100, 200, 300, 500, 1000, np.inf]
BAND_LABELS = ["0-99", "100-199", "200-299", "300-499", "500-999", "1000+"]

band_frames = []
for set_name in ("core", "full"):
    actual = actuals["all"]
    predicted = stored_predictions(set_name, "all")
    frame = pd.DataFrame(
        {
            "feature_set": set_name,
            "price_aud": actual,
            "abs_error": np.abs(predicted - actual),
        }
    )
    frame["band"] = pd.cut(frame["price_aud"], bins=PRICE_BANDS,
                           labels=BAND_LABELS, right=False)
    band_frames.append(frame)
bands = pd.concat(band_frames, ignore_index=True)

band_summary = (
    bands.groupby(["band", "feature_set"], observed=False)
    .agg(n=("abs_error", "size"),
         mean_abs_error=("abs_error", "mean"),
         median_abs_error=("abs_error", "median"))
    .reset_index()
)
print(f"absolute error by price band, unfiltered test set "
      f"(upper fence {fences['upper_aud']:,.0f} {CURRENCY})")
print()
print(
    band_summary.pivot(index="band", columns="feature_set",
                       values=["n", "mean_abs_error", "median_abs_error"])
    .round(2)
    .to_string()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True)
for axis, metric, label in (
    (axes[0], "mean_abs_error", "mean absolute error"),
    (axes[1], "median_abs_error", "median absolute error"),
):
    pivoted = band_summary.pivot(index="band", columns="feature_set", values=metric)
    pivoted[["core", "full"]].plot(kind="bar", ax=axis,
                                   color=["#4C72B0", "#55A868"], width=0.78)
    axis.set_ylabel(f"{label} ({CURRENCY})")
    axis.set_xlabel(f"actual price band ({CURRENCY})")
    axis.set_title(label.capitalize())
    axis.tick_params(axis="x", rotation=30)
    axis.legend(title="feature set", fontsize=8)
fig.suptitle("Absolute error by price band, unfiltered test set")
fig.tight_layout()
save_figure(fig, "model_error_by_price_band")
plt.show()

In [ ]:
GROUP_COLOURS = {
    "location": "#4C72B0",
    "performance": "#55A868",
    "demand": "#C44E52",
    "profile_other": "#8172B3",
    "space": "#CCB974",
    "other": "#937860",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))
for axis, set_name in zip(axes, ("core", "full")):
    table = importances[set_name].head(10).iloc[::-1]
    colours = [GROUP_COLOURS.get(g, "#937860") for g in table["group"]]
    axis.barh(table["feature"], table["share_pct"], color=colours)
    axis.set_xlabel("share of total importance (%)")
    axis.set_title(f"{set_name}: {winners[set_name]['model']}\n"
                   f"{importances[set_name].attrs['kind']}")
    axis.tick_params(labelsize=8)
present = set(importances["core"]["group"]) | set(importances["full"]["group"])
labels = [g for g in GROUP_COLOURS if g in present]
handles = [plt.Rectangle((0, 0), 1, 1, color=GROUP_COLOURS[g]) for g in labels]
fig.legend(handles, labels, loc="lower center", ncol=len(labels), fontsize=9,
           frameon=False, bbox_to_anchor=(0.5, -0.04))
fig.suptitle("Top 10 feature importances")
fig.tight_layout()
save_figure(fig, "model_feature_importances")
plt.show()

fig, axis = plt.subplots(figsize=(8, 4.4))
shares = group_shares["share_pct"].sort_values()
axis.barh(shares.index, shares.to_numpy(),
          color=[GROUP_COLOURS.get(g, "#937860") for g in shares.index])
for position, value in enumerate(shares.to_numpy()):
    axis.text(value + 0.6, position, f"{value:.1f}%", va="center", fontsize=9)
axis.set_xlabel("share of total importance (%)")
axis.set_title(f"Where the full model's fit comes from "
               f"({winners['full']['model']})")
axis.set_xlim(0, shares.max() * 1.18)
fig.tight_layout()
save_figure(fig, "model_importance_groups")
plt.show()

print()
print("reports/figures/ now holds:")
for path in sorted(FIGURE_DIR.glob("*.png")):
    print(f"  {path.name}  ({path.stat().st_size / 1024:.0f} KB)")

## 8. Run history: three failures and what changed

Four attempts produced these numbers. Recording why the first three failed is
the difference between a result and a reproducible result.

### Run 1: a hand-picked `max_features`, and a handicapped Random Forest

Random Forest was run with `max_features="sqrt"`, chosen by hand on speed alone
before any score was seen. The reason was real: scikit-learn's
`RandomForestRegressor` defaults to `max_features=1.0`, so every tree evaluates
all 88 encoded columns at every split, and the 540 fits that implies measured
**220.6 minutes** of serial fit time, over budget before XGBoost had run.

A probe then scored the three candidates under the same 10-fold CV:

| max_features | CV log RMSE | wall |
| --- | --- | --- |
| 1.0 (default) | 0.3298 | 398.7 s |
| 0.33 (Breiman) | 0.3299 | 124.8 s |
| sqrt | 0.3458 | 48.7 s |

`0.33` matches the default to four decimal places at 3.2x the speed, while
`"sqrt"` costs **4.87%** CV log RMSE. The hand-picked value was the wrong
trade, and it mattered: on `core`, Random Forest had landed within 0.5% of
XGBoost, close enough that the handicap could have decided which model won.
Run 1's results are kept at `reports/model_results_v1_sqrt.json`, named for
that `max_features` value, and section 9 compares them.

The fix was to replace the hand-picked value with a rule: **adopt the cheapest
candidate whose CV log RMSE is within 1% of the best.** That rule selects
`0.33`, and would do so for any tolerance between about 0.1% and 4.8%. To be
explicit about the order of events, the rule was written **after** seeing the
probe's numbers. That does not compromise the held-out test set, because the
probe and the rule read 10-fold cross-validation on the training split only.
What it does mean is that the 1% threshold was chosen knowing roughly where the
candidates fell, so it is a reasoned threshold rather than an independent one.
This run reads the stored probe record rather than repeating it.

The XGBoost grid was also widened, because run 1 put `n_estimators` and
`max_depth` on their upper edges in all three feature sets. The grid first
asked for was `[1000, 1500, 2000] x [4, 6, 8] x [0.03, 0.05, 0.1]`, measured at
6.80x the original cost or 21.7 minutes extra, over the 20 minute allowance.
The grid used is `[700, 1200, 2000] x [4, 6, 8] x [0.03, 0.05, 0.1]`, measured
at 5.67x or 17.5 minutes extra, which keeps three values per parameter and
brackets every run 1 choice.

### Run 2: killed by memory, twice

Run 2 died with a `MemoryError` inside a joblib worker, surfaced as a
`PicklingError`, and nbconvert wrote nothing, so an hour of compute was lost.
The cause, measured rather than guessed:

| | size |
| --- | --- |
| One `max_depth=None` forest | **448 MB** (6.2M nodes) |
| One 2000-tree depth-8 XGBoost booster | 19 MB |
| All nine fitted pipelines held at once | **1.4 GB** |
| RAM free at the time | 1.55 GB of 7.9 GB |

The notebook had been keeping every fitted pipeline in memory for the later
sections. By the time `full`/RandomForest ran, that was 1.36 GB held plus
0.9 GB in two search workers. XGBoost was never the problem.

Streaming the pipelines to disk fixed that, but a second attempt died the same
way: free RAM had fallen to 0.51 GB because other applications had grown to
7.4 GB of 7.9 GB, and one forest alone needs more than that.

### Run 3: training moved out of the notebook

Training now lives in `scripts/train_experiments.py`, which:

- runs **one search at a time**, as 9 independent `(model, feature_set)` units
  plus the leakage check, ordered Ridge, then XGBoost, then Random Forest last,
  and within Random Forest cheapest feature set first, so a failure on the
  largest still leaves the smaller ones done;
- writes each unit's results, predictions and fitted pipeline to disk the moment
  it finishes, then deletes the objects and collects, so **at most one fitted
  model is ever in memory**;
- **skips any unit that already has a checkpoint**, so a crash costs one unit
  rather than the whole run, with `--force` to redo one;
- uses `n_jobs=1` by default, which matters more than it sounds. The first
  attempt at run 3 kept `n_jobs=2` for XGBoost on the reasoning that its
  boosters are only 19 MB, and it died with
  `BrokenProcessPool: A task has failed to un-serialize`. The model size was
  never the issue: joblib's loky backend spawns a fresh interpreter per worker,
  roughly 200 MB before it unpickles anything, and at 0.74 GB free the worker
  could not allocate enough to receive its task. A probe confirmed `n_jobs=2`
  works on the same machine with small data, so this was memory, not pickling.
  With `n_jobs=1` joblib uses its sequential backend, which forks nothing;
- **refuses to start a fit when free RAM is below 0.4 GB**, waiting a minute at
  a time for up to 30 minutes and then stopping cleanly with a log line rather
  than crashing;
- logs every unit's start, end, wall time and free RAM to
  `reports/training_progress.log`.

`max_depth=None` was dropped from the Random Forest grid for this run, since it
is what produced the 448 MB forests. That does less than it appears to: the
`max_depth=20, min_samples_leaf=1` corner still reaches **338 MB**, because a
depth-20 cap still allows a fully grown tree on 12,454 rows. The parameter that
actually drives forest size is `min_samples_leaf`, which at 5 brings the same
forest down to 71 MB. The grid keeps `min_samples_leaf=1` as specified, so
`n_jobs=1` is doing the real work of keeping this inside the memory budget.

Run 3 got seven of the nine units onto disk. It then failed in two ways at
once, which is run 4's subject.

### Run 4: the silent `nan`, and a machine with room to work

Run 3 ended with `RandomForest/core_plus_space` raising `MemoryError` during
its final refit, after **70 of its 120 CV fits had already failed**. The
0.4 GB RAM floor had not stopped it, because the machine had RAM free and was
out of **commit charge**: every allocation reserves commit before it touches a
page, so a fit can die while `psutil`'s free-RAM figure still looks adequate.

The worse problem was that the run had not said so. All three XGBoost units
reported `CV log RMSE nan +/- nan` and wrote checkpoints anyway. Two mistakes
compounded:

1. `GridSearchCV` defaults to `error_score=np.nan`. A fold that dies is
   recorded as `nan` instead of raising, so the run continues and every
   parameter mean containing that fold becomes `nan`. `XGBoost/full` lost
   **7 of 270 fits** this way.
2. The best index was taken with `np.argmin(-mean_test_score)`. NumPy's
   `argmin` does not skip `nan`: a `nan` compares false against everything, so
   it propagates and `argmin` returns the `nan` position.

Together those two mean the script selected **the one parameter combination
whose fits had failed**, then copied its `nan` mean and `nan` SD into the
checkpoint. The damage is not only the missing CV figure: the `best_params`
recorded for those units are the failed combination's, not the winner's, so
the refitted pipelines and the test metrics on disk belong to a combination
the search never endorsed. Those checkpoints cannot be patched with a number.
They have to be retrained, which is why `XGBoost/full` was deleted outright.

What changed in the script:

- **`error_score="raise"`** on every `GridSearchCV` and every
  `cross_validate`. A failed fit now stops its unit with a traceback instead of
  becoming a `nan`, and the other units still hold their checkpoints.
- The CV mean and SD are read at **`best_index_`**, which scikit-learn derives
  from `rank_test_score`. Ranking sorts `nan` last rather than first, so a
  failed combination can never win even if one somehow survives. The scorer is
  `neg_root_mean_squared_error`, so the sign is flipped once and **no further
  square root** is taken.
- The CV mean and SD are **asserted finite** before any file is written.
- Every file is written to a **temporary name and renamed on success**: the
  pipeline, then the predictions, then the checkpoint last, because the
  checkpoint is what the skip test reads. A unit that raises deletes its
  temporaries and writes nothing, so an interrupted run can no longer leave a
  half-written checkpoint for the next run to skip over.
- The memory guard now reads the **commit charge** as well as physical RAM,
  through `GlobalMemoryStatusEx`, and logs both with every progress line. It
  waits when RAM is below 0.4 GB **or** commit headroom is below 1 GB, a
  minute at a time for up to 30 minutes, then stops cleanly.
- **`--n-jobs`** (default 1) raises the search parallelism and the forest's
  own `n_jobs` together, with XGBoost floored at its working value, so the same
  script runs single-threaded here and parallel on a larger machine.
- **`--verify-only`** reads every checkpoint and reports whether its CV mean
  and SD are finite and whether the file is complete, training nothing. It is
  the check that would have caught the `nan` an hour after it happened rather
  than at the end of the run.
- **`--backfill-cv`** reruns 10-fold cross-validation at a finished unit's
  stored parameters, on the same folds and the same cleaned training rows, and
  writes only the CV mean and SD back. It repairs a CV figure cheaply, but it
  does not re-select parameters, so it is not a repair for the `argmin` bug.
- `--only` accepts `leakage` as well as a `Model/feature_set` unit, and
  `--dry-run` lists what each unit would do without fitting.

Finally, the run moved to a machine with more RAM and more cores. That is not a
fix for anything above: the `nan` would have been just as silent with 64 GB,
and the guard and the loud failures are what make the result trustworthy. The
larger machine only means `--n-jobs` above 1 is available, so the same nine
units finish in a fraction of the time.

This notebook fits nothing. It reads checkpoints and renders.

In [ ]:
print("training units, from the checkpoints")
print()
timing = pd.DataFrame(
    [
        {
            "model": model_name,
            "feature_set": set_name,
            "search_seconds": record["search_seconds"],
            "refit_seconds": record["refit_seconds"],
            "wall_minutes": round(record["wall_seconds"] / 60, 2),
            "artefact_mb": round(record["artefact_bytes"] / 1e6, 1),
            "n_jobs": record.get("n_jobs"),
            "free_gb_at_start": record["free_gb_at_start"],
            # Written from run 4 onwards. Older checkpoints read as NaN, which
            # is the honest value: that run did not measure the commit charge.
            "commit_gb_at_start": record.get("commit_gb_at_start"),
            "finished_at": record["finished_at"],
        }
        for (model_name, set_name), record in checkpoints.items()
    ]
)
timing["_set"] = pd.Categorical(timing["feature_set"], SET_ORDER, ordered=True)
timing["_model"] = pd.Categorical(timing["model"], MODEL_ORDER, ordered=True)
timing = (
    timing.sort_values(["_model", "_set"])
    .drop(columns=["_set", "_model"])
    .reset_index(drop=True)
)
print(timing.to_string(index=False))

total_minutes = float(timing["wall_minutes"].sum())
if leakage_record:
    total_minutes += leakage_record["wall_seconds"] / 60
print()
print(f"total training time : {total_minutes:.1f} minutes")
print()
print("by model")
print(
    timing.groupby("model", observed=True)["wall_minutes"].sum().round(1).to_string()
)
print()
print(f"largest artefact : {timing['artefact_mb'].max():,.1f} MB")
print(f"total artefacts  : {timing['artefact_mb'].sum():,.1f} MB")
print(f"lowest free RAM at a unit start: "
      f"{timing['free_gb_at_start'].min():.2f} GB")
if timing["commit_gb_at_start"].notna().any():
    print(f"lowest commit headroom at a unit start: "
          f"{timing['commit_gb_at_start'].min():.2f} GB")
if timing["n_jobs"].notna().any():
    print(f"n_jobs used: {sorted(timing['n_jobs'].dropna().unique().tolist())}")

# A unit whose CV figures were recomputed after the fact says so, and says
# what the stored figures were before the backfill.
backfilled = {
    f"{model_name}/{set_name}": record["cv_backfill"]
    for (model_name, set_name), record in checkpoints.items()
    if "cv_backfill" in record
}
if backfilled:
    print()
    print("units whose CV mean and SD were backfilled with --backfill-cv")
    for label, detail in backfilled.items():
        print(f"  {label:<30} was {detail['previous_mean']} "
              f"+/- {detail['previous_sd']}, took {detail['seconds']:.1f}s")
    print("  best_params and the test metrics in those checkpoints are "
          "unchanged by a backfill")

## 9. Run 1 against this run

Read from `reports/model_results_v1_sqrt.json` rather than retyped, so it
cannot drift.

Only Random Forest should move much, because `max_features` applies to nothing
else. Ridge should be identical to several decimal places, since neither change
touches it, which makes it a useful control: if Ridge moved, something about the
split or the pipeline would have changed by accident. XGBoost moves only where
the widened grid found a better setting.

In [ ]:
V1_PATH = REPORTS_DIR / "model_results_v1_sqrt.json"
run_comparison = {}

if not V1_PATH.exists():
    print(f"reports/{V1_PATH.name} not found, nothing to compare")
else:
    v1 = json.loads(V1_PATH.read_text(encoding="utf-8"))
    v1_results = pd.DataFrame(v1["results"])
    print(f"run 1 Random Forest max_features : "
          f"{v1['deviations']['rf_max_features']['used']}")
    print(f"run 3 Random Forest max_features : "
          f"{context['rf_max_features']['selected']}")
    print(f"run 1 XGBoost grid : {v1['grids']['XGBoost']}")

    merged = v1_results.merge(
        results, on=["model", "feature_set"], suffixes=("_v1", "_v3")
    )
    merged["cv_change"] = (
        merged["cv_rmse_log_mean_v3"] - merged["cv_rmse_log_mean_v1"]
    )
    merged["cv_change_pct"] = (
        100 * merged["cv_change"] / merged["cv_rmse_log_mean_v1"]
    )
    merged["mae_change_aud"] = (
        merged["mae_aud_typical_v3"] - merged["mae_aud_typical_v1"]
    )
    merged["_set"] = pd.Categorical(merged["feature_set"], SET_ORDER, ordered=True)
    merged["_model"] = pd.Categorical(merged["model"], MODEL_ORDER, ordered=True)
    merged = (
        merged.sort_values(["_model", "_set"])
        .drop(columns=["_set", "_model"])
        .reset_index(drop=True)
    )

    print()
    print("CV log RMSE, run 1 against run 3 (negative change = run 3 better)")
    print(
        merged[["model", "feature_set", "cv_rmse_log_mean_v1",
                "cv_rmse_log_mean_v3", "cv_change", "cv_change_pct",
                "mae_aud_typical_v1", "mae_aud_typical_v3", "mae_change_aud"]]
        .round(4)
        .to_string(index=False)
    )

    forest = merged.loc[merged["model"] == "RandomForest"]
    if len(forest):
        print()
        print("=" * 78)
        print("Random Forest: sqrt (run 1) against 0.33 plus the trimmed grid")
        print("=" * 78)
        for row in forest.itertuples(index=False):
            print(f"  {row.feature_set:<16} "
                  f"CV log RMSE {row.cv_rmse_log_mean_v1:.4f} -> "
                  f"{row.cv_rmse_log_mean_v3:.4f}  "
                  f"({row.cv_change:+.4f}, {row.cv_change_pct:+.2f}%)   "
                  f"typical MAE {row.mae_aud_typical_v1:>7,.2f} -> "
                  f"{row.mae_aud_typical_v3:>7,.2f} {CURRENCY} "
                  f"({row.mae_change_aud:+,.2f})")
        print()
        print(f"  mean CV log RMSE change: {forest['cv_change'].mean():+.4f} "
              f"({forest['cv_change_pct'].mean():+.2f}%)")
        print(f"  largest single change  : {forest['cv_change'].min():+.4f} on "
              f"{forest.loc[forest['cv_change'].idxmin(), 'feature_set']}")

    ridge = merged.loc[merged["model"] == "Ridge"]
    if len(ridge):
        print()
        print("Ridge, the control: neither change should touch it")
        for row in ridge.itertuples(index=False):
            print(f"  {row.feature_set:<16} "
                  f"{row.cv_rmse_log_mean_v1:.6f} -> "
                  f"{row.cv_rmse_log_mean_v3:.6f}  "
                  f"(change {row.cv_change:+.6f})")

    xgb = merged.loc[merged["model"] == "XGBoost"]
    if len(xgb):
        print()
        print("XGBoost, where the widened grid applies")
        for row in xgb.itertuples(index=False):
            print(f"  {row.feature_set:<16} "
                  f"{row.cv_rmse_log_mean_v1:.4f} -> "
                  f"{row.cv_rmse_log_mean_v3:.4f}  "
                  f"({row.cv_change:+.4f}, {row.cv_change_pct:+.2f}%)")

    v1_winners = {k: v["model"] for k, v in v1["winners"].items()}
    v3_winners = {k: row["model"] for k, row in winners.items()}
    changed = [k for k in v3_winners if v1_winners.get(k) != v3_winners[k]]
    print()
    print("winners")
    print(f"  run 1: {v1_winners}")
    print(f"  run 3: {v3_winners}")
    print(f"  changed: {changed or 'none'}")

    run_comparison = {
        "v1_file": f"reports/{V1_PATH.name}",
        "v1_rf_max_features": v1["deviations"]["rf_max_features"]["used"],
        "v3_rf_max_features": context["rf_max_features"]["selected"],
        "v1_xgboost_grid": v1["grids"]["XGBoost"],
        "rows": merged[
            ["model", "feature_set", "cv_rmse_log_mean_v1", "cv_rmse_log_mean_v3",
             "cv_change", "cv_change_pct", "mae_aud_typical_v1",
             "mae_aud_typical_v3", "mae_change_aud"]
        ].to_dict(orient="records"),
        "random_forest_mean_cv_change": float(forest["cv_change"].mean())
        if len(forest)
        else None,
        "v1_winners": v1_winners,
        "v3_winners": v3_winners,
        "winners_changed": changed,
    }

## 10. Save `reports/model_results.json`

Assembled from the checkpoints, so this file and the notebook can never
disagree. `reports/` is git-ignored, which makes it a regenerable artefact.

A smoke run writes `reports/model_results_smoke.json` and
`reports/figures_smoke/` instead, so it cannot overwrite a real result.

In [ ]:
model_results = {
    "generated_by": "notebooks/04_model_training.ipynb",
    "trained_by": "scripts/train_experiments.py",
    "assembled_from": f"reports/{CHECKPOINT_DIR.name}/",
    "profile": CONFIG.profile,
    "smoke": CONFIG.smoke,
    "extras_present": sorted(extras),
    "dataset": context["dataset"],
    "currency": CURRENCY,
    "seed": context["seed"],
    "cv": context["cv"],
    "scope": context["scope"],
    "split": context["split"],
    "fences": context["fences"],
    "feature_sets": context["feature_sets"],
    "rf_max_features": context["rf_max_features"],
    "grids": {
        model_name: {
            edge["parameter"]: edge["candidates"]
            for edge in checkpoints[(model_name, "core")]["grid_edges"]
        }
        for model_name in MODEL_ORDER
        if (model_name, "core") in checkpoints
    },
    "results": results.drop(columns=["artefact_mb"]).to_dict(orient="records"),
    "winners": {
        set_name: {
            "model": row["model"],
            "best_params": row["best_params"],
            "cv_rmse_log_mean": row["cv_rmse_log_mean"],
            "cv_rmse_log_sd": row["cv_rmse_log_sd"],
            "rmse_aud_typical": row["rmse_aud_typical"],
            "mae_aud_typical": row["mae_aud_typical"],
            "r2_typical": row["r2_typical"],
            "rmse_log_typical": row["rmse_log_typical"],
            "rmse_aud_all": row["rmse_aud_all"],
            "mae_aud_all": row["mae_aud_all"],
            "r2_all": row["r2_all"],
            "rmse_log_all": row["rmse_log_all"],
        }
        for set_name, row in winners.items()
    },
    "core_vs_full": core_vs_full.to_dict(orient="records"),
    "core_vs_core_plus_space": core_vs_space.to_dict(orient="records"),
    "cv_by_model_and_set": pivot.reset_index().to_dict(orient="records"),
    "grid_edges": edges.to_dict(orient="records"),
    "grid_edges_flagged": int(len(flagged)),
    "importances": {
        set_name: {
            "model": winners[set_name]["model"],
            "kind": table.attrs["kind"],
            "top_10": table.head(10)[
                ["rank", "feature", "source_column", "group", "importance",
                 "share_pct"]
            ].to_dict(orient="records"),
        }
        for set_name, table in importances.items()
    },
    "importance_groups": {
        "assignment": GROUPS,
        "full_model_shares_pct": group_shares["share_pct"].to_dict()
        if len(group_shares)
        else {},
        "headline": headline,
        "top_source_columns": by_source.head(15).to_dict(orient="records")
        if len(by_source)
        else [],
    },
    "host_leakage_check": {
        "method": leakage_record["method"] if leakage_record else None,
        "rows": leakage.to_dict(orient="records"),
        "gaps": leakage_gap.to_dict(orient="records"),
    },
    "error_by_price_band": band_summary.astype({"band": str}).to_dict(
        orient="records"
    ),
    "training_timing": timing.to_dict(orient="records"),
    "total_training_minutes": round(total_minutes, 2),
    "run_1_comparison": run_comparison,
    "artefacts": {
        "figures": sorted(f"reports/figures/{p.name}"
                          for p in FIGURE_DIR.glob("model_*.png")),
        "pipelines": {
            f"{model_name}|{set_name}": record["artefact"]
            for (model_name, set_name), record in checkpoints.items()
        },
        "checkpoints": sorted(
            f"reports/checkpoints/{p.name}" for p in CHECKPOINT_DIR.glob("*.json")
        ),
    },
    "cv_provenance": {
        (f"{model_name}|{set_name}"): {
            "source": record.get("cv_source", "unrecorded"),
            "backfilled": "cv_backfill" in record,
        }
        for (model_name, set_name), record in checkpoints.items()
    },
    "run_history": {
        "run_1": "max_features='sqrt' hand-picked on speed; the probe then "
                 "showed it cost 4.87% CV log RMSE against the default while "
                 "0.33 matched it at 3.2x the speed. Kept at "
                 "reports/model_results_v1_sqrt.json.",
        "run_2": "died twice with MemoryError: a 448 MB forest plus nine held "
                 "pipelines (1.4 GB) against 1.55 GB free, then 0.51 GB free "
                 "on the retry.",
        "run_3": "training moved to scripts/train_experiments.py: one search "
                 "at a time, checkpointed and resumable, n_jobs=1 so joblib's "
                 "sequential backend forks no worker, and a 0.4 GB memory "
                 "floor that waits rather than crashes. A first attempt with "
                 "XGBoost at n_jobs=2 died with BrokenProcessPool because the "
                 "loky worker interpreter itself could not be allocated.",
        "run_4": "run 3 had failed quietly: GridSearchCV defaulted to "
                 "error_score=nan, so dead folds became nan, and "
                 "np.argmin(-mean_test_score) returns the nan position rather "
                 "than skipping it, so the three XGBoost units selected the "
                 "combination whose fits had failed and stored its nan mean "
                 "and nan SD. XGBoost/full lost 7 of 270 fits; "
                 "RandomForest/core_plus_space lost 70 of 120 and then its "
                 "refit died of commit exhaustion with RAM still free. Fixed "
                 "with error_score=raise everywhere, the CV figures read at "
                 "best_index_ and asserted finite, write-ahead temporary files "
                 "renamed only on success, a commit-charge floor beside the "
                 "RAM floor, and --verify-only, --backfill-cv, --n-jobs and "
                 "--dry-run. Rerun on a larger machine with --n-jobs above 1.",
    },
}

# RESULTS_PATH is set in section 0 and carries a _smoke suffix on a smoke run.
with RESULTS_PATH.open("w", encoding="utf-8") as handle:
    json.dump(model_results, handle, indent=2, default=str)
print(f"written : reports/{RESULTS_PATH.name}")
print(f"size    : {RESULTS_PATH.stat().st_size:,} bytes")
print(f"keys    : {list(model_results)}")

## 11. Learning curve (extra)

Does the model want more data, or more features? Each winner is refitted at
nested host-grouped fractions of the training rows and scored on the same
typical test set. Whole hosts are added at a time, so a 20% subset cannot
contain part of a host whose other listings are still held back.

The number to read is the last step. A large gain from 80% to 100% means the
curve is still climbing and more mentors would buy accuracy. A small one means
the ceiling is the feature set, which is the more useful finding for a project
whose real data has not arrived: it says the proxy dataset is not the
constraint.

Written by `scripts/train_experiments.py --extras`. This section is skipped if
that has not been run.

In [ ]:
learning_curve = pd.DataFrame()

if "learning_curve" not in extras:
    print("no learning curve checkpoint; run the script with --extras")
else:
    record = extras["learning_curve"]
    learning_curve = pd.DataFrame(record["rows"])
    print(record["method"])
    print(f"models: {record['models']}")
    print()
    print(
        learning_curve[["feature_set", "model", "fraction", "train_rows",
                        "train_hosts", "rmse_log_typical", "mae_aud_typical",
                        "r2_typical"]]
        .round(4)
        .to_string(index=False)
    )

    print()
    print("gain from the first fraction to the last, and from the last step")
    for row in record["gains"]:
        print(f"  {row['feature_set']:<16} {row['model']:<14} "
              f"{row['rmse_log_at_20pct']:.4f} -> {row['rmse_log_at_100pct']:.4f}  "
              f"total {row['total_gain_pct']:+.2f}%  "
              f"last step {row['last_step_gain_pct']:+.2f}%")
    print()
    print("a small last step means the curve has flattened, so more rows of")
    print("this kind would buy little and the ceiling is the feature set")

    fig, axis = plt.subplots(figsize=(8.5, 5.0))
    for set_name, colour in (("core", "#4C72B0"), ("full", "#55A868")):
        subset = learning_curve.loc[
            learning_curve["feature_set"] == set_name
        ].sort_values("train_rows")
        if not len(subset):
            continue
        axis.plot(subset["train_rows"], subset["rmse_log_typical"],
                  marker="o", color=colour,
                  label=f"{set_name} ({subset['model'].iloc[0]})")
    axis.set_xlabel("training rows (whole hosts, nested subsets)")
    axis.set_ylabel("test log RMSE (typical test set)")
    axis.set_title("Learning curve: does more data help?")
    axis.legend()
    fig.tight_layout()
    save_figure(fig, "model_learning_curve")
    plt.show()

## 12. Out-of-fold residual quantiles (extra)

This is what the application shows a mentor. A single predicted price is a
false promise, so the API returns a range, and the range has to come from
measured error rather than a round number.

These are the 10th and 90th percentiles of the log residual for the core
winner, measured **out of fold** on the training rows: each row's prediction
comes from the nine folds that did not contain it, so no row helped build the
model that scored it, and the held-out test set is not spent on calibration.

Because the target is `log1p(price)`, a residual quantile is a multiplicative
factor on the price. The application applies them as
`expm1(predicted_log + q10_log)` and `expm1(predicted_log + q90_log)`, which is
the same as multiplying the predicted price by `q10_factor` and `q90_factor`.

The per-band table matters because the error is not constant across the price
range: a flat percentage band would be too wide at the bottom and too narrow at
the top.

In [ ]:
residuals = pd.DataFrame()

if "residuals" not in extras:
    print("no residual checkpoint; run the script with --extras")
else:
    record = extras["residuals"]
    low, high = record["quantiles"]
    print(f"{record['model']} on {record['feature_set']}, "
          f"{record['train_rows']:,} training rows")
    print(record["method"])
    print(f"used for : {record['used_for']}")
    print(f"apply as : {record['how_to_apply']}")

    overall = record["overall"]
    print()
    print("overall")
    print(f"  q{low:<3} log {overall[f'q{low}_log']:+.4f}   "
          f"factor x{overall[f'q{low}_factor']:.3f}")
    print(f"  q{high:<3} log {overall[f'q{high}_log']:+.4f}   "
          f"factor x{overall[f'q{high}_factor']:.3f}")
    print(f"  median  {overall['median_log']:+.4f}   "
          f"sd {overall['sd_log']:.4f}")
    span = overall[f"q{high}_factor"] / overall[f"q{low}_factor"]
    print(f"  the {high - low}% band spans a factor of {span:.2f} "
          f"on the predicted price")

    residuals = pd.DataFrame(record["by_price_band"])
    print()
    print("by price band")
    columns = [c for c in ("band", "n", f"q{low}_log", f"q{high}_log",
                           f"q{low}_factor", f"q{high}_factor", "median_log",
                           "note") if c in residuals.columns]
    print(residuals[columns].round(4).to_string(index=False))

    print()
    print("worked example, a predicted 200 " + CURRENCY)
    predicted_log = float(np.log1p(200))
    bottom = float(np.expm1(predicted_log + overall[f"q{low}_log"]))
    top = float(np.expm1(predicted_log + overall[f"q{high}_log"]))
    print(f"  the application would show {bottom:,.0f} to {top:,.0f} {CURRENCY}")
    print(f"  so {high - low}% of comparable listings fall inside that band")

## 13. Permutation importance (extra)

Section 5's importances are model-internal: gain for XGBoost, impurity
decrease for the forest, a standardised coefficient for Ridge. None of them is
comparable across model families and all of them are measured on the data the
model was fitted to.

Permutation importance asks a different question on held-out rows: how much
worse does this model get, in log RMSE, when this column is shuffled into
noise. The units are the metric's, the measurement is out of sample, and the
columns are the source columns rather than the one-hot encodings, so a
neighbourhood is one feature instead of sixty fragments.

A value at or below zero means the column carries nothing the model uses.

In [ ]:
permutation = {}

if "permutation" not in extras:
    print("no permutation importance checkpoint; run the script with --extras")
else:
    record = extras["permutation"]
    print(record["method"])
    print(f"scored on {record['test_rows']:,} typical test rows, "
          f"{record['n_repeats']} repeats")

    for set_name, payload in record["by_feature_set"].items():
        table = pd.DataFrame(payload["rows"])
        permutation[set_name] = table
        print()
        print("=" * 78)
        print(f"{set_name} winner: {payload['model']}")
        print("=" * 78)
        print(
            table.head(12)[["rank", "source_column", "group",
                            "rmse_log_increase", "sd", "share_pct"]]
            .round(5)
            .to_string(index=False)
        )
        useless = table.loc[table["rmse_log_increase"] <= 0, "source_column"]
        if len(useless):
            print(f"  carrying nothing the model uses ({len(useless)}): "
                  f"{', '.join(useless.tolist()[:10])}")
        print()
        print("  group shares of the total rise in log RMSE")
        for group, share in payload["group_shares_pct"].items():
            print(f"    {group:<14} {share:6.2f}%")

    if permutation:
        fig, axes = plt.subplots(
            1, len(permutation), figsize=(7.2 * len(permutation), 5.6), squeeze=False
        )
        for axis, (set_name, table) in zip(axes[0], permutation.items()):
            top = table.head(10).iloc[::-1]
            colours = [GROUP_COLOURS.get(g, "#937860") for g in top["group"]]
            axis.barh(top["source_column"], top["rmse_log_increase"],
                      xerr=top["sd"], color=colours, error_kw={"lw": 0.8})
            axis.set_xlabel("rise in log RMSE when shuffled")
            axis.set_title(f"{set_name}: "
                           f"{record['by_feature_set'][set_name]['model']}")
            axis.tick_params(labelsize=8)
        fig.suptitle("Permutation importance on the typical test set")
        fig.tight_layout()
        save_figure(fig, "model_permutation_importance")
        plt.show()

## 14. Standard against workstation profile

A profile widens the search grid and changes nothing else: the split, the
fences, the target, the folds and the metrics are identical, which is what
makes the two comparable. Each profile keeps its own checkpoint folder, so both
sets of numbers survive and the question can be answered rather than assumed.

Two things to read. First, did the wider grid actually buy accuracy, or did it
spend hours confirming the narrow grid was already at the optimum. Second, is
any best parameter still sitting on an edge of even the wider grid, which would
mean the search still wants to keep going and the reported optimum is a lower
bound.

This section is skipped unless both folders exist.

In [ ]:
profile_comparison = pd.DataFrame()

OTHER_PROFILE = "workstation" if CONFIG.profile == "standard" else "standard"
other_dir = te.RunConfig(profile=OTHER_PROFILE).checkpoints


def read_profile(directory):
    """The results table of whatever finished units a folder holds."""
    found = []
    for model_name, set_name in UNITS:
        path = directory / f"{model_name}__{set_name}.json".lower()
        if not path.exists():
            continue
        payload = json.loads(path.read_text(encoding="utf-8"))
        found.append(
            {
                "model": model_name,
                "feature_set": set_name,
                "cv_rmse_log_mean": payload.get("cv_rmse_log_mean"),
                "mae_aud_typical": payload["metrics"]["typical"]["mae_aud"],
                "best_params": payload["best_params"],
                "wall_minutes": round(payload["wall_seconds"] / 60, 2),
                "edges": [
                    e["parameter"] for e in payload["grid_edges"]
                    if "EDGE" in e["position"]
                ],
            }
        )
    return pd.DataFrame(found)


this_profile = read_profile(CHECKPOINT_DIR)
other = read_profile(other_dir) if other_dir.exists() else pd.DataFrame()

print(f"this run : {CONFIG.profile:<12} reports/{CHECKPOINT_DIR.name}  "
      f"({len(this_profile)} units)")
print(f"other    : {OTHER_PROFILE:<12} reports/{other_dir.name}  "
      f"({len(other)} units)")

if not len(other):
    print()
    print(f"nothing to compare: reports/{other_dir.name} has no finished units.")
    print(f"Run the script with --profile {OTHER_PROFILE} to produce them.")
else:
    profile_comparison = this_profile.merge(
        other, on=["model", "feature_set"],
        suffixes=(f"_{CONFIG.profile}", f"_{OTHER_PROFILE}"),
    )
    this_cv = f"cv_rmse_log_mean_{CONFIG.profile}"
    other_cv = f"cv_rmse_log_mean_{OTHER_PROFILE}"
    profile_comparison["cv_change"] = (
        profile_comparison[this_cv] - profile_comparison[other_cv]
    )
    profile_comparison["cv_change_pct"] = (
        100 * profile_comparison["cv_change"] / profile_comparison[other_cv]
    )
    profile_comparison["_set"] = pd.Categorical(
        profile_comparison["feature_set"], SET_ORDER, ordered=True
    )
    profile_comparison["_model"] = pd.Categorical(
        profile_comparison["model"], MODEL_ORDER, ordered=True
    )
    profile_comparison = (
        profile_comparison.sort_values(["_model", "_set"])
        .drop(columns=["_set", "_model"])
        .reset_index(drop=True)
    )

    print()
    print(f"CV log RMSE, {CONFIG.profile} against {OTHER_PROFILE} "
          f"(negative change = {CONFIG.profile} better)")
    print(
        profile_comparison[["model", "feature_set", other_cv, this_cv,
                            "cv_change", "cv_change_pct"]]
        .round(4)
        .to_string(index=False)
    )

    better = int((profile_comparison["cv_change"] < 0).sum())
    print()
    print(f"did the wider grid buy anything? {CONFIG.profile} is better on "
          f"{better} of {len(profile_comparison)} units")
    print(f"  mean change      : {profile_comparison['cv_change'].mean():+.4f} "
          f"({profile_comparison['cv_change_pct'].mean():+.2f}%)")
    print(f"  best improvement : "
          f"{profile_comparison['cv_change'].min():+.4f} on "
          f"{profile_comparison.loc[profile_comparison['cv_change'].idxmin(), 'model']}"
          f"/{profile_comparison.loc[profile_comparison['cv_change'].idxmin(), 'feature_set']}")
    this_minutes = profile_comparison[f"wall_minutes_{CONFIG.profile}"].sum()
    other_minutes = profile_comparison[f"wall_minutes_{OTHER_PROFILE}"].sum()
    print(f"  cost             : {this_minutes:,.1f} minutes against "
          f"{other_minutes:,.1f}, a factor of "
          f"{this_minutes / other_minutes if other_minutes else float('nan'):.1f}")

    still_on_edge = profile_comparison.loc[
        profile_comparison[f"edges_{CONFIG.profile}"].str.len() > 0
    ]
    print()
    if len(still_on_edge):
        print(f"best parameters still on a grid edge in the {CONFIG.profile} "
              f"profile ({len(still_on_edge)} units):")
        for row in still_on_edge.itertuples(index=False):
            chosen = getattr(row, f"best_params_{CONFIG.profile}")
            edges = getattr(row, f"edges_{CONFIG.profile}")
            print(f"  {row.model:<14} {row.feature_set:<16} "
                  f"{', '.join(edges)}  chose {chosen}")
        print()
        print("an edge means even the wider grid did not bracket the optimum,")
        print("so that model's tuned score is still a lower bound")
    else:
        print(f"no best parameter sits on an edge in the {CONFIG.profile} "
              f"profile: every grid bracketed its optimum")

## 15. Verification

The checks that would catch a mistake in any of the above.

In [ ]:
print("verification")
print()

print(f"  search units with checkpoints : {len(checkpoints)} of {len(UNITS)}")
assert len(checkpoints) == len(UNITS), f"missing units: {missing}"
print(f"  leakage unit present          : {leakage_record is not None}")
assert leakage_record is not None, "the leakage unit has no checkpoint"

assert split["hosts_in_both"] == 0
print("  no host in both train and test: True")

# Smoke mode keeps a seeded tenth of the rows on both sides, so these counts
# are deliberately a fraction of the split's and cannot reconcile against it.
# The subsample is recorded in the context, so this is a check being skipped
# for a stated reason rather than a check quietly not applying.
if fences.get("smoke_subsample"):
    print(f"  row reconciliation           : skipped, smoke subsample "
          f"{fences['smoke_subsample']:.2f} of the rows")
else:
    assert (fences["training_rows_used"]
            + fences["rows_outside"]["train"]["outside"]
            == split["train_rows"])
    print("  training rows reconcile       : True")
    assert (fences["test_rows_typical"] + fences["rows_outside"]["test"]["outside"]
            == fences["test_rows_all"])
    print("  test rows reconcile           : True")

assert len(results) == len(UNITS)
print(f"  results rows                  : {len(results)}")

# Run 3 wrote three checkpoints whose CV mean and SD were nan, and nothing
# downstream noticed: a nan sorts out of idxmin, so the affected model simply
# never appeared as a winner. This check is what turns that into a failure.
not_finite = results.loc[
    ~np.isfinite(results["cv_rmse_log_mean"])
    | ~np.isfinite(results["cv_rmse_log_sd"]),
    ["model", "feature_set", "cv_rmse_log_mean", "cv_rmse_log_sd"],
]
assert not len(not_finite), (
    "non-finite CV figures, so a search failed silently:\n"
    + not_finite.to_string(index=False)
)
print("  every CV mean and SD finite   : True")

excluded = {"id", "host_id", "price_num", "long_stay", "latitude", "longitude"}
for name, columns in context["feature_sets"].items():
    leaked = excluded & set(columns)
    assert not leaked, f"{name} uses an identifier, the target or a dropped column: {leaked}"
print("  no identifier, target or dropped column used as a feature: True")

for key, record in checkpoints.items():
    path = PROJECT_ROOT / record["artefact"]
    assert path.exists(), f"missing artefact for {key}"
total_mb = sum(r["artefact_bytes"] for r in checkpoints.values()) / 1e6
print(f"  every fitted pipeline on disk : True ({total_mb:,.0f} MB)")

print()
print(f"  this notebook fitted nothing: every number came from "
      f"reports/{CHECKPOINT_DIR.name}/")
if CONFIG.smoke:
    print()
    print("  SMOKE RUN: 3-fold CV on a tenth of the rows with single-value")
    print("  grids. Every number above proves the pipeline executes and")
    print("  nothing more. Do not report any of it.")

print()
print("final summary")
for set_name in SET_ORDER:
    if set_name not in winners:
        continue
    row = winners[set_name]
    print(f"  {set_name:<16} {row['model']:<14} "
          f"CV log RMSE {row['cv_rmse_log_mean']:.4f}   "
          f"typical MAE {row['mae_aud_typical']:,.2f} {CURRENCY}")
if {"core", "full"} <= set(winners):
    gap = winners["full"]["cv_rmse_log_mean"] - winners["core"]["cv_rmse_log_mean"]
    print()
    print(f"  full against core: {gap:+.4f} log RMSE "
          f"({100 * gap / winners['core']['cv_rmse_log_mean']:+.2f}%)")
if headline:
    print(f"  full model fit   : location {headline['location']:.1f}%, "
          f"performance {headline['performance']:.1f}%, "
          f"demand {headline['demand']:.1f}%")